In [ ]:
##Make sure that an Ollama session is running first in the terminal:
## ollama run qwen3.5:2b

## In my case is running on http://localhost:11434/



In [1]:
import os
# Ollama server address
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
#os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]

In [2]:
# ================= Excel Batch Processing (Optimized) =================
import os


In [3]:
import random

# ------------------- 1-Shot Example -------------------
VALID_1_SHOT_EXAMPLE = """
{
    "A": ["B", "C"],
    "B": ["D"],
    "C": [],
    "D": []
}
""".strip()

# ------------------- Prompt Generator -------------------
def extraction_prompt(graph_type: str, min_nodes: int = 3, max_nodes: int = 6) -> str:
    num_nodes = random.randint(min_nodes, max_nodes)

    prompt = f"""Generate a BFS input graph in Python as an adjacency list.

Here is a valid example:
{VALID_1_SHOT_EXAMPLE}

Now generate ONE graph with EXACTLY {num_nodes} nodes.

The graph MUST represent this case: {graph_type}

Definitions:
- valid: a correct connected graph. No missing nodes, no cycles, connected

Rules:
- Use string node names like "A", "B", etc.
- Values must be lists
- Output MUST be a valid Python dictionary (except for invalid structure case)
- Do NOT include explanations
- Return ONLY the dictionary

Output:"""

    return prompt.strip()

In [4]:
# ------------------- System Prompt -------------------
system_prompt = """
You are an expert in undergrad CS topics. Your task is to generate BFS graph inputs following strict formats.

You must:
- Output ONLY a Python dictionary
- Do NOT include explanations, comments, or text outside the dictionary
- Use adjacency list format: { "A": ["B", "C"], ... }
- Node names must be strings
- Values must be lists

Follow the user instructions EXACTLY, especially the requested graph type.

Output only the graph.
""".strip()

In [5]:
from collections import deque

def bfs(graph, start="A"):
    if not isinstance(graph, dict):
        raise ValueError("Invalid graph structure")
    if start not in graph:
        raise KeyError(f"Start node '{start}' not found")

    visited = set()
    queue = deque([start])
    order = []

    while queue:
        node = queue.popleft()

        if node in visited:
            continue

        if node not in graph:
            raise KeyError(f"Node '{node}' missing in graph")

        visited.add(node)
        order.append(node)

        neighbors = graph[node]
        if not isinstance(neighbors, list):
            raise ValueError(f"Neighbors of '{node}' must be a list")

        for neighbor in neighbors:
            queue.append(neighbor)

    return order

In [6]:
import asyncio
import ollama

MODEL_NAME = "codellama:7b"
TEMPERATURE = 0.5

async def call_llm_async(prompt: str, temperature: float = 0.0, timeout: int = 10) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    async def _call():
        response = await ollama.AsyncClient().chat(
            model=MODEL_NAME,
            messages=messages,
            options={"temperature": temperature}
        )
        content = (response.get("message", {}).get("content", "") or "").strip()
        return content if content else "ERROR: Empty response"

    try:
        return await asyncio.wait_for(_call(), timeout=timeout)
    except asyncio.TimeoutError:
        return "TIMEOUT"
    except Exception as e:
        return f"ERROR: {str(e)}"

In [7]:
import ast

def parse_graph(graph_text: str):
    try:
        return ast.literal_eval(graph_text), "OK"
    except Exception as e:
        return None, f"ERROR: {e}"

In [8]:
def run_bfs_test(graph, graph_type="valid"):
    result_record = {
        "graph": graph,
        "behavior": None,
        "error": None,
        "expected_failure": graph_type
    }

    try:
        if not graph:
            raise ValueError("Empty graph")

        start_node = list(graph.keys())[0]
        traversal = bfs(graph, start_node)

        result_record["behavior"] = "OK"
        result_record["traversal_order"] = traversal

    except Exception as e:
        result_record["behavior"] = "Error"
        result_record["error"] = str(e)

    return result_record

In [9]:
prompt = extraction_prompt(graph_type = "valid")

In [10]:
graph_text = await call_llm_async(prompt)

In [11]:
print(graph_text)

{
    "A": ["B", "C"],
    "B": ["D"],
    "C": [],
    "D": ["E"],
    "E": ["F"],
    "F": []
}


In [12]:
parsed_graph, parse_status = parse_graph(graph_text)

In [13]:
llm_test_output = run_bfs_test(parsed_graph, graph_type = "valid")

In [14]:
print(llm_test_output)

{'graph': {'A': ['B', 'C'], 'B': ['D'], 'C': [], 'D': ['E'], 'E': ['F'], 'F': []}, 'behavior': 'OK', 'error': None, 'expected_failure': 'valid', 'traversal_order': ['A', 'B', 'C', 'D', 'E', 'F']}
